In [1]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
torchaudio.datasets.SPEECHCOMMANDS(root="./data", download=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [4]:
def extract_mfcc(path, max_len=100):
    y, sr = librosa.load(path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < max_len:
        pad = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad)))
    else:
        mfcc = mfcc[:, :max_len]

    return mfcc.T

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, root, cache_dir="mfcc_cache"):
        cache_path = os.path.join(root, cache_dir)
        mfcc_path = os.path.join(cache_path, "mfccs.dat")
        label_path = os.path.join(cache_path, "labels.npy")
        meta_path = os.path.join(cache_path, "meta.npy")

        assert os.path.exists(mfcc_path), (
            f"Cache not found at {cache_path}. Run `python precompute_mfcc.py` first!"
        )
        shape = tuple(np.load(meta_path))
        self.classes = list(np.load(os.path.join(cache_path, "classes.npy")))
        self.files = list(np.load(os.path.join(cache_path, "files.npy")))

        self.mfccs = np.memmap(mfcc_path, dtype='float32', mode='r', shape=shape)
        self.label_arr = np.load(label_path)

        print(f"Loaded {len(self.files)} cached MFCCs (memory-mapped)")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mfcc = np.array(self.mfccs[idx])
        return torch.tensor(mfcc, dtype=torch.float32), int(self.label_arr[idx])

In [ ]:
root = "./data/SpeechCommands/speech_commands_v0.02"

val_list_path = os.path.join(root, "validation_list.txt")
test_list_path = os.path.join(root, "testing_list.txt")

val_files = set()
test_files = set()

if os.path.exists(val_list_path):
    with open(val_list_path) as f:
        val_files = set(line.strip() for line in f)
if os.path.exists(test_list_path):
    with open(test_list_path) as f:
        test_files = set(line.strip() for line in f)

dataset = SpeechDataset(root)
NUM_CLASSES = len(dataset.classes)

train_idx, val_idx, test_idx = [], [], []
for i, filepath in enumerate(dataset.files):
    rel = os.path.relpath(filepath, root).replace("\\", "/")
    if rel in test_files:
        test_idx.append(i)
    elif rel in val_files:
        val_idx.append(i)
    else:
        train_idx.append(i)

train_ds = torch.utils.data.Subset(dataset, train_idx)
val_ds   = torch.utils.data.Subset(dataset, val_idx)
test_ds  = torch.utils.data.Subset(dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, num_workers=0, pin_memory=True)

print(f"Classes ({NUM_CLASSES}): {dataset.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Loaded 105829 cached MFCCs (memory-mapped)
Classes (35): ['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']
Train: 84843, Val: 9981, Test: 11005


In [7]:
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(40, 64, 3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)

        _, (h, _) = self.lstm(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        out = self.fc(h)
        return out

In [8]:
model = CNN_LSTM(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [9]:
EPOCHS = 15
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for x, y in loop:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

        loop.set_postfix(loss=loss.item())

    train_acc = correct / total
    scheduler.step()

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            val_correct += (pred == y).sum().item()
            val_total += y.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, LR={scheduler.get_last_lr()[0]:.6f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  -> Saved best model (Val Acc={val_acc:.4f})")

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Epoch 1/15: 100%|██████████| 1326/1326 [00:25<00:00, 52.05it/s, loss=0.636]


Epoch 1: Train Acc=0.6873, Val Acc=0.8621, LR=0.001000
  -> Saved best model (Val Acc=0.8621)


Epoch 2/15: 100%|██████████| 1326/1326 [00:22<00:00, 59.15it/s, loss=0.306]


Epoch 2: Train Acc=0.8669, Val Acc=0.9025, LR=0.001000
  -> Saved best model (Val Acc=0.9025)


Epoch 3/15: 100%|██████████| 1326/1326 [00:22<00:00, 58.26it/s, loss=0.246] 


...

Epoch 13: Train Acc=0.9777, Val Acc=0.9359, LR=0.000250


Epoch 14/15: 100%|██████████| 1326/1326 [00:33<00:00, 39.50it/s, loss=0.0815] 


Epoch 14: Train Acc=0.9786, Val Acc=0.9366, LR=0.000250


Epoch 15/15: 100%|██████████| 1326/1326 [00:31<00:00, 42.13it/s, loss=0.0989] 


Epoch 15: Train Acc=0.9808, Val Acc=0.9348, LR=0.000125

Best Val Accuracy: 0.9395


In [10]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        pred = out.argmax(1)
        correct += (pred.cpu() == y).sum().item()
        total += y.size(0)

print(f"Test Accuracy: {correct/total:.4f}")

Test Accuracy: 0.9250


In [11]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        pred = out.argmax(1).cpu().tolist()
        all_preds.extend(pred)
        all_labels.extend(y.tolist())

precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=dataset.classes))

Precision: 0.9259
Recall:    0.9250
F1 Score:  0.9251

              precision    recall  f1-score   support

    backward       0.90      0.94      0.92       165
         bed       0.88      0.86      0.87       207
        bird       0.91      0.89      0.90       185
         cat       0.93      0.94      0.94       194
         dog       0.88      0.90      0.89       220
        down       0.90      0.91      0.91       406
       eight       0.94      0.95      0.94       408
        five       0.93      0.96      0.95       445
      follow       0.88      0.83      0.86       172
     forward       0.77      0.85      0.81       155
        four       0.92      0.87      0.90       400
          go       0.87      0.88      0.87       402
       happy       0.93      0.94      0.93       203
       house       0.96      0.95      0.96       191
       learn       0.77      0.84      0.81       161
        left       0.96      0.94      0.95       412
      marvin       0.93   